In [1]:
# ============================================================
# EXERCISE 1: Manual TF-IDF Vectorization
# ============================================================

import math
import string
from collections import Counter

# ----------------------------------------------------------
# STEP 1: Tokenization and Pre-processing
# ----------------------------------------------------------

corpus_raw = [
    "The Cat chased the Mouse.",       # D1
    "The dog Barked at the Cat.",      # D2
    "The mouse ran away from the Cat." # D3
]

def preprocess(text):
    text = text.lower()                          # lowercase
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    tokens = text.split()                        # tokenize
    return tokens

tokenized_docs = [preprocess(doc) for doc in corpus_raw]

print("=" * 50)
print("STEP 1: Tokenized Documents")
print("=" * 50)
for i, tokens in enumerate(tokenized_docs, 1):
    print(f"D{i}: {tokens}")


# ----------------------------------------------------------
# STEP 2: Vocabulary Construction (sorted alphabetically)
# ----------------------------------------------------------

all_words = [word for doc in tokenized_docs for word in doc]
vocabulary = sorted(set(all_words))

print("\n" + "=" * 50)
print("STEP 2: Vocabulary (sorted)")
print("=" * 50)
print(vocabulary)


# ----------------------------------------------------------
# STEP 3: Term Frequency (TF) Calculation
# ----------------------------------------------------------
# Formula: TF(t, d) = f(t,d) / sum(f(k,d))  for all k in d

def compute_tf(tokens, vocabulary):
    total_terms = len(tokens)
    freq = Counter(tokens)
    tf = {word: freq.get(word, 0) / total_terms for word in vocabulary}
    return tf

tf_matrix = [compute_tf(doc, vocabulary) for doc in tokenized_docs]

print("\n" + "=" * 50)
print("STEP 3: Term Frequency (TF) Matrix")
print("=" * 50)

# Header
header = f"{'Term':<12}" + "".join(f"{'D'+str(i):<10}" for i in range(1, 4))
print(header)
print("-" * 40)

for word in vocabulary:
    row = f"{word:<12}" + "".join(f"{tf_matrix[i][word]:<10.4f}" for i in range(3))
    print(row)


# ----------------------------------------------------------
# STEP 3.1: Normalized TF
# ----------------------------------------------------------
# Using log normalization: w_tf = 1 + log10(f(t,d)) if f(t,d)>0, else 0

def compute_normalized_tf(tokens, vocabulary):
    freq = Counter(tokens)
    wtf = {}
    for word in vocabulary:
        f = freq.get(word, 0)
        wtf[word] = 1 + math.log10(f) if f > 0 else 0
    return wtf

normalized_tf_matrix = [compute_normalized_tf(doc, vocabulary) for doc in tokenized_docs]

print("\n" + "=" * 50)
print("STEP 3.1: Normalized TF (log normalization)")
print("=" * 50)

header = f"{'Term':<12}" + "".join(f"{'D'+str(i):<10}" for i in range(1, 4))
print(header)
print("-" * 40)

for word in vocabulary:
    row = f"{word:<12}" + "".join(f"{normalized_tf_matrix[i][word]:<10.4f}" for i in range(3))
    print(row)


# ----------------------------------------------------------
# STEP 4: Inverse Document Frequency (IDF) Calculation
# ----------------------------------------------------------
# Formula: IDF(t) = log10( N / (1 + df_t) )

N = len(tokenized_docs)

def compute_idf(vocabulary, tokenized_docs):
    idf = {}
    for word in vocabulary:
        df = sum(1 for doc in tokenized_docs if word in doc)
        idf[word] = math.log10(N / (1 + df))
    return idf

idf_values = compute_idf(vocabulary, tokenized_docs)

print("\n" + "=" * 50)
print("STEP 4: IDF Values")
print("=" * 50)
for word, val in idf_values.items():
    print(f"  IDF({word:<10}) = {val:.4f}")


# ----------------------------------------------------------
# STEP 5: TF-IDF Score Computation
# ----------------------------------------------------------
# Formula: TF-IDF(t, d) = TF(t, d) * IDF(t)

print("\n" + "=" * 50)
print("STEP 5: TF-IDF Matrix")
print("=" * 50)

header = f"{'Term':<12}" + "".join(f"{'D'+str(i):<12}" for i in range(1, 4))
print(header)
print("-" * 50)

tfidf_matrix = []
for i, tf in enumerate(tf_matrix):
    tfidf_doc = {word: tf[word] * idf_values[word] for word in vocabulary}
    tfidf_matrix.append(tfidf_doc)

for word in vocabulary:
    row = f"{word:<12}" + "".join(f"{tfidf_matrix[i][word]:<12.4f}" for i in range(3))
    print(row)


# ----------------------------------------------------------
# Discussion Answers
# ----------------------------------------------------------
print("\n" + "=" * 50)
print("DISCUSSION ANSWERS")
print("=" * 50)
print("""
Q1: Why TF-IDF instead of raw TF?
    Raw TF gives high scores to common words (like "the")
    that appear in every document but carry no meaning.
    TF-IDF penalizes these common words via IDF, making
    rare but informative words stand out more.

Q2: Which terms get higher TF-IDF importance?
    Terms that appear frequently in ONE document but
    rarely across all documents get the highest scores
    (e.g., "chased", "barked", "ran").

Q3: Effect of adding more documents?
    As N increases, IDF values shift. Words appearing
    in most documents approach IDF → 0, while unique
    words get even more weight.
""")

STEP 1: Tokenized Documents
D1: ['the', 'cat', 'chased', 'the', 'mouse']
D2: ['the', 'dog', 'barked', 'at', 'the', 'cat']
D3: ['the', 'mouse', 'ran', 'away', 'from', 'the', 'cat']

STEP 2: Vocabulary (sorted)
['at', 'away', 'barked', 'cat', 'chased', 'dog', 'from', 'mouse', 'ran', 'the']

STEP 3: Term Frequency (TF) Matrix
Term        D1        D2        D3        
----------------------------------------
at          0.0000    0.1667    0.0000    
away        0.0000    0.0000    0.1429    
barked      0.0000    0.1667    0.0000    
cat         0.2000    0.1667    0.1429    
chased      0.2000    0.0000    0.0000    
dog         0.0000    0.1667    0.0000    
from        0.0000    0.0000    0.1429    
mouse       0.2000    0.0000    0.1429    
ran         0.0000    0.0000    0.1429    
the         0.4000    0.3333    0.2857    

STEP 3.1: Normalized TF (log normalization)
Term        D1        D2        D3        
----------------------------------------
at          0.0000    1.0000    

In [2]:
# ============================================================
# EXERCISE 2: CBOW - Data Generation
# ============================================================

# ----------------------------------------------------------
# STEP 1: Vocabulary and Indexing
# ----------------------------------------------------------

# After stop-word removal
documents = {
    "D1": ["cat", "chased", "mouse"],
    "D2": ["dog", "barked", "cat"],
    "D3": ["mouse", "ran", "cat"]
}

# Sorted unique vocabulary
vocab = sorted(set(word for doc in documents.values() for word in doc))
print("=" * 50)
print("Vocabulary:", vocab)
print("=" * 50)

# Assign index to each word
word2idx = {word: idx for idx, word in enumerate(vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

# One-hot vector function
def one_hot(index, vocab_size):
    vec = [0] * vocab_size
    vec[index] = 1
    return vec

V = len(vocab)  # vocabulary size = 6

print("\nWord Index Mapping and One-Hot Vectors:")
print(f"{'Word':<10} {'Index':<8} {'One-Hot Vector'}")
print("-" * 40)
for word in vocab:
    idx = word2idx[word]
    oh = one_hot(idx, V)
    print(f"{word:<10} {idx:<8} {oh}")


# ----------------------------------------------------------
# STEP 2: Look-Up Table with Context-Target Pairs
# Window size = 1 (one word on each side)
# ----------------------------------------------------------

def generate_context_target_pairs(doc_tokens, window_size=1):
    pairs = []
    for i, target in enumerate(doc_tokens):
        context = []
        for j in range(i - window_size, i + window_size + 1):
            if j != i and 0 <= j < len(doc_tokens):
                context.append(doc_tokens[j])
        if context:
            pairs.append((context, target))
    return pairs

print("\n" + "=" * 50)
print("STEP 2: Context-Target Pairs (Look-Up Tables)")
print("=" * 50)

for doc_name, tokens in documents.items():
    pairs = generate_context_target_pairs(tokens, window_size=1)
    print(f"\n{doc_name}: {tokens}")
    print(f"{'Context (Words)':<20} {'Context (Index)':<20} {'Target Word':<14} {'Target (Index)'}")
    print("-" * 75)
    for context_words, target_word in pairs:
        context_idx = [word2idx[w] for w in context_words]
        target_idx = word2idx[target_word]
        print(f"{str(context_words):<20} {str(context_idx):<20} {target_word:<14} {target_idx}")


# ----------------------------------------------------------
# Discussion Answer
# ----------------------------------------------------------
print("\n" + "=" * 50)
print("DISCUSSION ANSWERS")
print("=" * 50)
print("""
Q: How does CBOW handle 'cat' appearing in multiple documents?
   CBOW processes each (context, target) pair independently
   during training. Since "cat" appears with different context
   words in each document, its embedding is updated multiple
   times — each update nudges the vector toward capturing
   ALL contexts where "cat" appears.

Q: Connection to the Distributional Hypothesis?
   The Distributional Hypothesis states: "words that occur
   in similar contexts have similar meanings." CBOW directly
   implements this — it trains word embeddings by predicting
   a target word from its surrounding context words. Words
   sharing similar contexts will end up with similar
   embedding vectors.
""")

Vocabulary: ['barked', 'cat', 'chased', 'dog', 'mouse', 'ran']

Word Index Mapping and One-Hot Vectors:
Word       Index    One-Hot Vector
----------------------------------------
barked     0        [1, 0, 0, 0, 0, 0]
cat        1        [0, 1, 0, 0, 0, 0]
chased     2        [0, 0, 1, 0, 0, 0]
dog        3        [0, 0, 0, 1, 0, 0]
mouse      4        [0, 0, 0, 0, 1, 0]
ran        5        [0, 0, 0, 0, 0, 1]

STEP 2: Context-Target Pairs (Look-Up Tables)

D1: ['cat', 'chased', 'mouse']
Context (Words)      Context (Index)      Target Word    Target (Index)
---------------------------------------------------------------------------
['chased']           [2]                  cat            1
['cat', 'mouse']     [1, 4]               chased         2
['chased']           [2]                  mouse          4

D2: ['dog', 'barked', 'cat']
Context (Words)      Context (Index)      Target Word    Target (Index)
---------------------------------------------------------------------------
['ba

In [3]:
# ============================================================
# EXERCISE 3.2: CBOW Architecture Design and Forward Pass
# ============================================================

import numpy as np

# ----------------------------------------------------------
# STEP 1: Define Architecture Dimensions
# ----------------------------------------------------------

V = 6   # Vocabulary size
N = 4   # Hidden layer (embedding) dimension

print("=" * 50)
print("STEP 1: CBOW Architecture")
print("=" * 50)
print(f"  Input Layer  : {V} neurons  (one per vocabulary word)")
print(f"  Hidden Layer : {N} neurons  (embedding dimension)")
print(f"  Output Layer : {V} neurons  (one per vocabulary word)")


# ----------------------------------------------------------
# STEP 2: Weight Matrix Dimensions and Random Initialization
# ----------------------------------------------------------

print("\n" + "=" * 50)
print("STEP 2: Weight Matrices")
print("=" * 50)

# W_input: shape (V x N)  — maps one-hot input to embedding
W_input = np.array([
    [ 0.1,  0.2, -0.1,  0.0],   # index 0 → barked
    [ 0.4,  0.5, -0.1,  0.2],   # index 1 → cat
    [-0.2,  0.3,  0.2, -0.1],   # index 2 → chased
    [ 0.3, -0.1,  0.1,  0.0],   # index 3 → dog
    [ 0.5, -0.4,  0.1,  0.1],   # index 4 → mouse
    [-0.3,  0.0, -0.2,  0.3],   # index 5 → ran
])

print(f"\nW_input  shape: {W_input.shape}  (V x N = {V} x {N})")
print("W_input =")
print(W_input)

# W_hidden: shape (N x V)  — maps hidden layer to output scores
W_hidden = np.array([
    [ 0.2, -0.1,  0.3,  0.1,  0.4,  0.0],
    [ 0.1,  0.3, -0.2, -0.1,  0.2,  0.5],
    [-0.4,  0.5,  0.1, -0.2, -0.1,  0.3],
    [ 0.1, -0.2, -0.4,  0.3, -0.3,  0.1],
])

print(f"\nW_hidden shape: {W_hidden.shape}  (N x V = {N} x {V})")
print("W_hidden =")
print(W_hidden)


# ----------------------------------------------------------
# STEP 3: Computations at the Hidden Layer
# Context pair: ["cat", "mouse"]  →  Target: "chased"
# ----------------------------------------------------------

print("\n" + "=" * 50)
print("STEP 3: Hidden Layer Computation")
print("CONTEXT: ['cat', 'mouse']  |  TARGET: 'chased'")
print("=" * 50)

# One-hot vectors for context words
cat_one_hot   = np.array([0, 1, 0, 0, 0, 0])  # index 1
mouse_one_hot = np.array([0, 0, 0, 0, 1, 0])  # index 4

# Project each context word → selects corresponding row from W_input
V1_hat = cat_one_hot @ W_input     # row 1 of W_input
V2_hat = mouse_one_hot @ W_input   # row 4 of W_input

print(f"\nProjection for 'cat'   (index 1): V1_hat = {V1_hat}")
print(f"Projection for 'mouse' (index 4): V2_hat = {V2_hat}")

# Average context vector
V_hat = (V1_hat + V2_hat) / 2
print(f"\nAverage Context Vector V_hat = {V_hat}")


# ----------------------------------------------------------
# Project to Output Layer: Z = V_hat @ W_hidden
# ----------------------------------------------------------

print("\n" + "=" * 50)
print("STEP 3 (cont.): Project to Output Layer")
print("=" * 50)

Z = V_hat @ W_hidden
print(f"Z (logits) = V_hat @ W_hidden")
print(f"Z = {Z}")


# ----------------------------------------------------------
# STEP 4: Softmax → Probability Distribution
# ----------------------------------------------------------

def softmax(z):
    e_z = np.exp(z - np.max(z))
    return e_z / e_z.sum()

probs = softmax(Z)

print("\n" + "=" * 50)
print("STEP 4: Output Layer — Softmax Probabilities")
print("=" * 50)
print(f"{'Word':<10} {'Logit Z_i':<14} {'P(word | context)'}")
print("-" * 40)

vocab = ["barked", "cat", "chased", "dog", "mouse", "ran"]
for i, word in enumerate(vocab):
    marker = " ← predicted target" if i == np.argmax(probs) else ""
    print(f"{word:<10} {Z[i]:<14.4f} {probs[i]:.4f}{marker}")

predicted_word = vocab[np.argmax(probs)]
print(f"\nPredicted word: '{predicted_word}'")
print(f"True target   : 'chased'")
print(f"\n(Note: Weights are random, so prediction may be wrong before training.)")


# ----------------------------------------------------------
# Architecture Summary
# ----------------------------------------------------------
print("\n" + "=" * 50)
print("ARCHITECTURE SUMMARY")
print("=" * 50)
print(f"""
  Layer          Size    Description
  ─────────────────────────────────────────────────
  Input Layer    {V}       One-hot vectors for context words
  W_input        {V}×{N}    Embedding matrix (learned)
  Hidden Layer   {N}       Average of projected context embeddings
  W_hidden       {N}×{V}    Output weight matrix (discarded after training)
  Output Layer   {V}       Softmax probabilities over vocabulary
  ─────────────────────────────────────────────────
  After training, only W_input is kept as word embeddings.
""")

STEP 1: CBOW Architecture
  Input Layer  : 6 neurons  (one per vocabulary word)
  Hidden Layer : 4 neurons  (embedding dimension)
  Output Layer : 6 neurons  (one per vocabulary word)

STEP 2: Weight Matrices

W_input  shape: (6, 4)  (V x N = 6 x 4)
W_input =
[[ 0.1  0.2 -0.1  0. ]
 [ 0.4  0.5 -0.1  0.2]
 [-0.2  0.3  0.2 -0.1]
 [ 0.3 -0.1  0.1  0. ]
 [ 0.5 -0.4  0.1  0.1]
 [-0.3  0.  -0.2  0.3]]

W_hidden shape: (4, 6)  (N x V = 4 x 6)
W_hidden =
[[ 0.2 -0.1  0.3  0.1  0.4  0. ]
 [ 0.1  0.3 -0.2 -0.1  0.2  0.5]
 [-0.4  0.5  0.1 -0.2 -0.1  0.3]
 [ 0.1 -0.2 -0.4  0.3 -0.3  0.1]]

STEP 3: Hidden Layer Computation
CONTEXT: ['cat', 'mouse']  |  TARGET: 'chased'

Projection for 'cat'   (index 1): V1_hat = [ 0.4  0.5 -0.1  0.2]
Projection for 'mouse' (index 4): V2_hat = [ 0.5 -0.4  0.1  0.1]

Average Context Vector V_hat = [0.45 0.05 0.   0.15]

STEP 3 (cont.): Project to Output Layer
Z (logits) = V_hat @ W_hidden
Z = [ 0.11  -0.06   0.065  0.085  0.145  0.04 ]

STEP 4: Output Layer — Softmax